# 02. Data Preparation & Feature Engineering Pipeline
## Supply Chain Optimization — FMCG / Retail

---

### Objective
1. Validate incoming data schema and check for data leakage risks.
2. Execute missing value analysis and outlier treatment using `DataPreprocessor`.
3. Engineer core domain features:
   - **Demand-to-Supply Ratio (DSR)**: $\text{Demand} / \text{Supply}$
   - **Lead-Time Variability Index**: $\sigma_L / L$
   - **Stockout Severity Frequency Score (SSFS)**
   - **Safety Stock Coverage & Inventory Days of Supply (DOS)**
   - **Reorder & Shortage Gaps**


In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
from src.preprocessing import DataPreprocessor, aggregate_warehouse_profiles
from src.feature_engineering import SupplyChainFeatureEngineer, get_feature_documentation


### 1. Ingestion and Schema Validation


In [ ]:
df_raw = pd.read_csv("../data/raw/fmcg_supply_chain_raw.csv")
preprocessor = DataPreprocessor(impute_strategy="median", handle_outliers=True)

valid, issues = preprocessor.validate_schema(df_raw)
print("Schema Validated:", valid)
if not valid:
    print("Schema Issues:", issues)


### 2. Preprocessing & Outlier Treatment


In [ ]:
df_cleaned = preprocessor.fit_transform(df_raw)
print(f"Cleaned Data Dimensions: {df_cleaned.shape}")


### 3. Advanced Feature Engineering


In [ ]:
fe = SupplyChainFeatureEngineer(benchmark_dsr=1.0)
df_engineered = fe.fit_transform(df_cleaned)

# Save processed dataset
df_engineered.to_csv("../data/processed/fmcg_supply_chain_engineered.csv", index=False)
print("Engineered Features Count:", len(fe.get_feature_names()))
df_engineered[fe.get_feature_names()].head()


### 4. Feature Dictionary Documentation


In [ ]:
feature_docs = get_feature_documentation()
pd.DataFrame.from_dict(feature_docs, orient="index")


### 5. Warehouse Profile Aggregation
Aggregate transaction history into static facility-level profiles for clustering.


In [ ]:
df_profiles = aggregate_warehouse_profiles(df_engineered)
df_profiles.to_csv("../data/processed/warehouse_profiles.csv", index=False)
print(f"Aggregated Profiles for {len(df_profiles)} Facilities.")
df_profiles.head()
